In [ ]:
Requirement already satisfied: opencv-python in /usr/local/lib/python3.10/dist-packages (4.10.0.84)
Requirement already satisfied: numpy in /usr/local/lib/python3.10/dist-packages (1.26.4)
Requirement already satisfied: matplotlib in /usr/local/lib/python3.10/dist-packages (3.8.0)
Requirement already satisfied: contourpy>=1.0.1 in /usr/local/lib/python3.10/dist-packages (from matplotlib) (1.3.0)
Requirement already satisfied: cycler>=0.10 in /usr/local/lib/python3.10/dist-packages (from matplotlib) (0.12.1)
Requirement already satisfied: fonttools>=4.22.0 in /usr/local/lib/python3.10/dist-packages (from matplotlib) (4.54.1)
Requirement already satisfied: kiwisolver>=1.0.1 in /usr/local/lib/python3.10/dist-packages (from matplotlib) (1.4.7)
Requirement already satisfied: packaging>=20.0 in /usr/local/lib/python3.10/dist-packages (from matplotlib) (24.1)
Requirement already satisfied: pillow>=6.2.0 in /usr/local/lib/python3.10/dist-packages (from matplotlib) (10.4.0)
Requirement already satisfied: pyparsing>=2.3.1 in /usr/local/lib/python3.10/dist-packages (from matplotlib) (3.2.0)
Requirement already satisfied: python-dateutil>=2.7 in /usr/local/lib/python3.10/dist-packages (from matplotlib) (2.8.2)
Requirement already satisfied: six>=1.5 in /usr/local/lib/python3.10/dist-packages (from python-dateutil>=2.7->matplotlib) (1.16.0)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Cargar la imagen médica (en escala de grises)
image = cv2.imread('/content/radiografia.jpg', cv2.IMREAD_GRAYSCALE)

# Mostrar la imagen original
plt.figure(figsize=(10, 6))
plt.subplot(2, 3, 1)
plt.imshow(image, cmap='gray')
plt.title('Imagen Original')
plt.axis('off')

# Preprocesamiento: convertir la imagen a binaria usando umbralización
_, binary_image = cv2.threshold(image, 127, 255, cv2.THRESH_BINARY)

# Crear un kernel para las operaciones morfológicas
kernel = np.ones((7, 7), np.uint8)

# Operaciones morfológicas
# Apertura: elimina el ruido (elimina pequeñas manchas en el fondo)
opening = cv2.morphologyEx(binary_image, cv2.MORPH_OPEN, kernel)

# Cierre: llena los agujeros dentro de los objetos (en este caso, podría ser útil para objetos como tumores o células)
closing = cv2.morphologyEx(opening, cv2.MORPH_CLOSE, kernel)

# Mostrar las imágenes después de las operaciones morfológicas
plt.subplot(2, 3, 2)
plt.imshow(opening, cmap='gray')
plt.title('Apertura')
plt.axis('off')

plt.subplot(2, 3, 3)
plt.imshow(closing, cmap='gray')
plt.title('Cierre')
plt.axis('off')

# Detener el procesamiento si la imagen ya está limpia

# Detección de contornos para obtener los objetos segmentados
contours, _ = cv2.findContours(closing, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# Dibujar los contornos en la imagen original para visualización
contour_image = cv2.drawContours(np.copy(image), contours, -1, (0, 255, 0), 2)

# Mostrar los contornos
plt.subplot(2, 3, 4)
plt.imshow(contour_image, cmap='gray')
plt.title('Contornos detectados')
plt.axis('off')

# Extraer características geométricas: área y perímetro
areas = []
perimeters = []
for contour in contours:
    area = cv2.contourArea(contour)
    perimeter = cv2.arcLength(contour, True)  # 'True' porque es un contorno cerrado
    areas.append(area)
    perimeters.append(perimeter)

# Mostrar estadísticas
print("Áreas de los objetos detectados:", areas)
print("Perímetros de los objetos detectados:", perimeters)

# Mostrar los valores de área y perímetro en el documento
plt.subplot(2, 3, 5)
plt.imshow(closing, cmap='gray')
plt.title('Resultados finales')
plt.axis('off')

# Mostrar la gráfica con los valores de área y perímetro
plt.subplot(2, 3, 6)
plt.plot(areas, label="Áreas", color="blue")
plt.plot(perimeters, label="Perímetros", color="red")
plt.legend()
plt.title('Características de los objetos')
plt.xlabel('Objeto indexado')
plt.ylabel('Valor')
plt.tight_layout()
plt.show()